# 🎵 Music AI Classifier: Обучение модели классификации жанра и вайба

Этот ноутбук обучает двухголовую свёрточную нейросеть **MusicAudioNet**:
- **Голова 1 (Жанры, 10 классов):** Blues, Classical, Country, Disco, Hiphop, Jazz, Metal, Pop, Reggae, Rock
- **Голова 2 (Вайб, 3 класса):** ⚡ Драйв/Энергия, ☕ Чилл/Отдых, 🧠 Фокус/Концентрация

Результат: сжатая модель **`music_classifier.int8.onnx` (~1.2 МБ)**, готовая для Rust-плеера.

In [ ]:
!pip install -q torchaudio onnx onnxruntime
import os, sys, time, json, tarfile, random, urllib.request
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")

## 1. Архитектура модели (Dual-Head ResNet-Audio)

In [ ]:
GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]
VIBES = ["Драйв / Энергия", "Чилл / Отдых", "Фокус / Концентрация"]

GENRE_TO_VIBE = {
    "disco": 0, "hiphop": 0, "metal": 0, "rock": 0,
    "blues": 1, "country": 1, "pop": 1, "reggae": 1,
    "classical": 2, "jazz": 2,
}

class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.act1 = nn.LeakyReLU(0.1, inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.act2 = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x):
        return self.act2(self.bn2(self.conv2(self.act1(self.bn1(self.conv1(x))))) + self.shortcut(x))

class MusicAudioNet(nn.Module):
    def __init__(self, num_genres: int = len(GENRES), num_vibes: int = len(VIBES)):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=(2, 2), padding=2, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1, inplace=True)
        )
        self.stage1 = ConvBlock(32, 64, stride=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.stage2 = ConvBlock(64, 96, stride=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.stage3 = ConvBlock(96, 128, stride=1)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.stage4 = ConvBlock(128, 192, stride=1)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.3)
        
        self.genre_head = nn.Sequential(
            nn.Linear(192, 96), nn.LeakyReLU(0.1, inplace=True), nn.Dropout(0.2), nn.Linear(96, num_genres)
        )
        self.vibe_head = nn.Sequential(
            nn.Linear(192, 48), nn.LeakyReLU(0.1, inplace=True), nn.Linear(48, num_vibes)
        )

    def forward(self, mel):
        x = self.stem(mel)
        x = self.pool1(self.stage1(x))
        x = self.pool2(self.stage2(x))
        x = self.pool3(self.stage3(x))
        x = self.stage4(x)
        feat = self.dropout(self.global_pool(x).flatten(1))
        return self.genre_head(feat), self.vibe_head(feat)

m = MusicAudioNet()
print(f"Параметров модели: {sum(p.numel() for p in m.parameters()):,}")

## 2. Загрузка и подготовка датасета GTZAN

In [ ]:
SAMPLE_RATE = 16000
CHUNK_DURATION = 10.0
CHUNK_SAMPLES = int(SAMPLE_RATE * CHUNK_DURATION)
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 512

data_dir = Path("/kaggle/working/data")
data_dir.mkdir(parents=True, exist_ok=True)
genres_dir = data_dir / "genres"

if not genres_dir.exists() or len(list(genres_dir.glob("*/*.wav"))) < 900:
    tar_path = data_dir / "genres.tar.gz"
    if not tar_path.exists():
        print("Скачиваем GTZAN (1.2 GB)...")
        urllib.request.urlretrieve("http://opihi.cs.uvic.ca/sound/genres.tar.gz", str(tar_path))
    print("Распаковываем...")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=data_dir)
print(f"Найдено треков: {len(list(genres_dir.glob('*/*.wav')))}")

# Сплит 80/20 с нарезкой по 3 сэмпла на трек в обучении
random.seed(42)
train_slices, val_slices = [], []
for g_idx, g_name in enumerate(GENRES):
    files = sorted(list((genres_dir / g_name).glob("*.wav")))
    random.shuffle(files)
    split = int(len(files) * 0.8)
    v_idx = GENRE_TO_VIBE[g_name]
    for f in files[:split]:
        for s_idx in (0, 1, 2):
            train_slices.append((f, s_idx, g_idx, v_idx))
    for f in files[split:]:
        val_slices.append((f, 1, g_idx, v_idx))

random.shuffle(train_slices); random.shuffle(val_slices)
print(f"Обучающих сэмплов: {len(train_slices)}, Валидационных: {len(val_slices)}")

## 3. Dataset и SpecAugment Аугментация

In [ ]:
class MusicDataset(Dataset):
    def __init__(self, slices, is_train=True):
        self.slices = slices
        self.is_train = is_train
        self.mel_tf = T.MelSpectrogram(sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS, power=2.0)
        self.amp_db = T.AmplitudeToDB()
        self.freq_mask = T.FrequencyMasking(16)
        self.time_mask = T.TimeMasking(32)

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        f_path, s_idx, g_idx, v_idx = self.slices[idx]
        try:
            waveform, sr = torchaudio.load(str(f_path))
        except:
            waveform, sr = torch.zeros(1, CHUNK_SAMPLES), SAMPLE_RATE
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sr != SAMPLE_RATE:
            waveform = T.Resample(sr, SAMPLE_RATE)(waveform)
        start = s_idx * CHUNK_SAMPLES
        end = start + CHUNK_SAMPLES
        if waveform.shape[1] < end:
            waveform = F.pad(waveform, (0, end - waveform.shape[1]))
        chunk = waveform[:, start:end]
        with torch.no_grad():
            mel = self.amp_db(self.mel_tf(chunk))
            mel = (mel - mel.mean()) / (mel.std() + 1e-6)
            if self.is_train:
                mel = self.time_mask(self.freq_mask(mel))
        return mel, torch.tensor(g_idx), torch.tensor(v_idx)

train_loader = DataLoader(MusicDataset(train_slices, True), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(MusicDataset(val_slices, False), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

## 4. Обучение на GPU (30 эпох с AMP FP16)

In [ ]:
model = MusicAudioNet().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30, eta_min=1e-5)
crit_g = nn.CrossEntropyLoss(label_smoothing=0.05)
crit_v = nn.CrossEntropyLoss(label_smoothing=0.05)
scaler = torch.amp.GradScaler(enabled=(DEVICE == "cuda"))

best_acc = 0.0
best_weights = Path("/kaggle/working/best_music_model.pth")

for epoch in range(1, 31):
    model.train()
    t0 = time.time()
    tr_loss, tr_g_corr, tr_v_corr, total = 0.0, 0, 0, 0
    for mel, g_tgt, v_tgt in train_loader:
        mel, g_tgt, v_tgt = mel.to(DEVICE), g_tgt.to(DEVICE), v_tgt.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=(DEVICE == "cuda")):
            lg, lv = model(mel)
            loss = crit_g(lg, g_tgt) + 0.5 * crit_v(lv, v_tgt)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        tr_loss += loss.item() * mel.size(0)
        tr_g_corr += (lg.argmax(-1) == g_tgt).sum().item()
        tr_v_corr += (lv.argmax(-1) == v_tgt).sum().item()
        total += mel.size(0)
    sched.step()
    
    # Val
    model.eval()
    vl_g_corr, vl_v_corr, vl_total = 0, 0, 0
    with torch.no_grad():
        for mel, g_tgt, v_tgt in val_loader:
            mel, g_tgt, v_tgt = mel.to(DEVICE), g_tgt.to(DEVICE), v_tgt.to(DEVICE)
            lg, lv = model(mel)
            vl_g_corr += (lg.argmax(-1) == g_tgt).sum().item()
            vl_v_corr += (lv.argmax(-1) == v_tgt).sum().item()
            vl_total += mel.size(0)
            
    val_g_acc = (vl_g_corr / vl_total) * 100
    val_v_acc = (vl_v_corr / vl_total) * 100
    print(f"Epoch {epoch:02d}/30 [{time.time()-t0:.1f}s] | Train Genre: {tr_g_corr/total*100:.1f}%, Vibe: {tr_v_corr/total*100:.1f}% | Val Genre: {val_g_acc:.1f}%, Vibe: {val_v_acc:.1f}%")
    if val_g_acc > best_acc:
        best_acc = val_g_acc
        torch.save(model.state_dict(), best_weights)
        print(f"  >>> Лучшая точность: {best_acc:.2f}%")

## 5. Экспорт в ONNX (FP32 и сжатие в INT8) + labels.json для Rust

In [ ]:
export_dir = Path("/kaggle/working/export")
export_dir.mkdir(exist_ok=True)
fp32_onnx = export_dir / "music_classifier.onnx"
int8_onnx = export_dir / "music_classifier.int8.onnx"

model.load_state_dict(torch.load(best_weights, map_location="cpu"))
model.eval().cpu()

dummy = torch.randn(1, 1, N_MELS, 313, dtype=torch.float32)
torch.onnx.export(
    model, dummy, str(fp32_onnx),
    input_names=["mel_spectrogram"], output_names=["genre_logits", "vibe_logits"],
    dynamic_axes={"mel_spectrogram": {0: "batch", 3: "time"}, "genre_logits": {0: "batch"}, "vibe_logits": {0: "batch"}},
    opset_version=17
)

quantize_dynamic(str(fp32_onnx), str(int8_onnx), weight_type=QuantType.QInt8)

metadata = {
    "model_name": "MusicAudioNet",
    "sample_rate": SAMPLE_RATE,
    "chunk_duration_sec": CHUNK_DURATION,
    "chunk_samples": CHUNK_SAMPLES,
    "n_mels": N_MELS,
    "n_fft": N_FFT,
    "hop_length": HOP_LENGTH,
    "genres": GENRES,
    "vibes": VIBES,
    "genre_to_vibe": GENRE_TO_VIBE,
}
with open(export_dir / "labels.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("\nРазмеры файлов в export/:")
for f in export_dir.iterdir():
    print(f"  • {f.name}: {f.stat().st_size / (1024*1024):.2f} MB")

!zip -j -r /kaggle/working/music_model_export.zip /kaggle/working/export/*
print("\nГотово! Скачайте файл music_model_export.zip из вкладки Output справа.")